# Day 2 — Gemma 4 Recovery Data Generation

**Research:** *Learning to Recover: Teaching Small Coding Agents to Fix Their Own Mistakes Using Execution Feedback*

### What this notebook does

1. Reuse the **actual** scripts and MBPP split manifest saved in Day 1.
2. Load Gemma 4 E4B in 4-bit on Kaggle T4.
3. Sample multiple solutions for **training** problems, execute the existing **visible** tests, and retry after failure.
4. Record every trajectory incrementally; rerunning an interrupted collection cell **skips completed trajectories as long as the checkpoint JSONL remains available**. After a Kaggle session reset, restore the saved JSONL from a backup before resuming.
5. Construct and verify C1 success-only and C2 recovery datasets, separating genuine Gemma self-recoveries from reference-based corrections.
6. Produce a downloadable `gemma_day2_backup.zip`.

**Important:** This uses the same subprocess-based test runner as Day 1, which is **not a secure sandbox**. Run only in a disposable environment **without secrets or sensitive files**. Never feed hidden MBPP+ tests to the model.

**Before running:** Create a **new Kaggle notebook**, select **GPU T4 ×2**, enable Internet for package/model/dataset downloads, then use **Add Input → Notebooks** to attach your Day 1 saved notebook outputs. Alternatively upload `gemma_day1_backup.zip` as a private Kaggle Dataset and attach that Dataset. The next cells find either input automatically.

## A — Dependencies

In [ ]:
%pip install -q -U transformers accelerate bitsandbytes datasets

The familiar pip warnings about `pyarrow`, `bigframes`, or a possible restart may appear. If imports and model loading work, you do not need to restart.

## B — Recover your Day 1 scripts and split manifest from Kaggle Input

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import json
import shutil
import sys

SRC = Path('/kaggle/working/day2_src')
SRC.mkdir(parents=True, exist_ok=True)
required = [
    'prepare_datasets.py', 'test_runner.py', 'agent_loop.py',
    'mbpp_split_manifest.json'
]

input_root = Path('/kaggle/input')
zip_candidates = sorted(input_root.rglob('gemma_day1_backup*.zip')) if input_root.exists() else []
source_description = None

for candidate in zip_candidates:
    try:
        with ZipFile(candidate) as z:
            names = set(z.namelist())
            if set(required).issubset(names):
                for name in required:
                    (SRC / name).write_bytes(z.read(name))
                source_description = str(candidate)
                break
    except Exception as exc:
        print('Skipping unreadable ZIP:', candidate, str(exc)[:200])

if source_description is None and input_root.exists():
    # Saved notebook output may expose the files directly instead of a ZIP.
    for manifest_path in sorted(input_root.rglob('mbpp_split_manifest.json')):
        parent = manifest_path.parent
        if all((parent / name).is_file() for name in required):
            for name in required:
                shutil.copy2(parent / name, SRC / name)
            source_description = str(parent)
            break

if source_description is None:
    raise FileNotFoundError(
        'Day 1 output was not attached. Use Kaggle Add Input → Notebooks → '
        'your Google - Gemma 4 (Paper Track) notebook; or upload and attach '
        'gemma_day1_backup.zip as a private Kaggle Dataset.'
    )

sys.path.insert(0, str(SRC))
print('Day 1 source:', source_description)
print('Restored:', [p.name for p in SRC.iterdir()])
manifest = json.loads((SRC / 'mbpp_split_manifest.json').read_text())
print('Eligible train IDs:', manifest['mbppplus_decontaminated']['train']['count'])
print('Reserved MBPP+ evaluation IDs:', len(manifest['evalplus_mbppplus_ids']))

## C — Day 2 helper module (data validation and checkpointing)

In [ ]:
%%writefile /kaggle/working/day2_src/day2_dataset.py
"""Day 2 trajectory extraction and SFT dataset construction.

Adapted to the user's Day 1 scripts:
  prepare_datasets.py, agent_loop.py, test_runner.py.

Generated programs are executed via Day 1's resource-limited subprocess runner.
THIS IS NOT A SECURITY SANDBOX. Do not run with secrets or sensitive files.
"""
from __future__ import annotations

import hashlib
import json
import os
import random
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Callable

from agent_loop import build_initial_messages, run_agent
from test_runner import run_visible_tests


def read_jsonl(path: str | Path) -> list[dict]:
    path = Path(path)
    if not path.exists():
        return []
    rows = []
    for line_no, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
        if line.strip():
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"Broken JSONL at {path}:{line_no}; restore the last checkpoint") from exc
    return rows


def write_jsonl(path: str | Path, rows: list[dict]) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as stream:
        for row in rows:
            stream.write(json.dumps(row, ensure_ascii=False) + "\n")


def select_training_tasks(eligible: list[dict], seed: int, limit: int) -> list[dict]:
    unique = {int(row["task_id"]): row for row in eligible}
    if len(unique) != len(eligible):
        raise ValueError("Duplicate training task IDs")
    ids = sorted(unique)
    random.Random(seed).shuffle(ids)
    return [unique[i] for i in ids[:min(len(ids), limit)]]


def collect_trajectories(
    problems: list[dict],
    generate_fn: Callable,
    checkpoint_path: str | Path,
    model_id: str,
    seed_fn: Callable[[int], None] | None = None,
    seed: int = 42,
    samples_per_task: int = 2,
    max_attempts: int = 3,
    max_new_tokens: int = 384,
) -> list[dict]:
    """Run initial generation + feedback-conditioned attempts; append every trajectory.

    Safe to rerun with same settings: keys already completed are skipped.
    """
    if not 1 <= max_attempts <= 3:
        raise ValueError("Supported retry budget: 1..3 total attempts")
    checkpoint_path = Path(checkpoint_path)
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    prior = read_jsonl(checkpoint_path)
    # Checkpoints with incompatible configurations should not be reused silently.
    for row in prior:
        if (row.get("model") != model_id or row.get("generation_seed_base") != seed or
            row.get("max_attempts") != max_attempts or row.get("max_new_tokens") != max_new_tokens):
            raise ValueError("Checkpoint configuration differs from this run. Use a new output path.")
    completed = {(int(r["task_id"]), int(r["sample_index"])) for r in prior}
    remaining = sum((int(p["task_id"]), i) not in completed for p in problems for i in range(samples_per_task))
    print(f"Existing trajectories: {len(prior)} | New trajectories to generate: {remaining}", flush=True)
    for p in problems:
        task_id = int(p["task_id"])
        for sample_idx in range(samples_per_task):
            key = (task_id, sample_idx)
            if key in completed:
                continue
            trajectory_seed = seed + task_id * 1009 + sample_idx
            if seed_fn is not None:
                seed_fn(trajectory_seed)
            row = run_agent(p, generate_fn, max_attempts=max_attempts, max_new_tokens=max_new_tokens)
            row.update({
                "model": model_id,
                "split": "train",
                "sample_index": sample_idx,
                "trajectory_seed": trajectory_seed,
                "generation_seed_base": seed,
                "max_attempts": max_attempts,
                "max_new_tokens": max_new_tokens,
                "created_utc": datetime.now(timezone.utc).isoformat(),
            })
            with checkpoint_path.open("a", encoding="utf-8") as out:
                out.write(json.dumps(row, ensure_ascii=False) + "\n")
                out.flush()
                os.fsync(out.fileno())
            prior.append(row)
            completed.add(key)
            statuses = " -> ".join(h["status"] for h in row["history"])
            print(f"task={task_id} sample={sample_idx}: {statuses}; total new tokens="
                  f"{row['total_completion_tokens']}", flush=True)
    return prior


def _retry_text(feedback: str) -> str:
    # Keep *exactly* the user retry instruction from Day 1 agent_loop.py.
    return ("The previous code FAILED the visible tests.\n" + feedback +
            "\nProvide a corrected full Python solution. Do not repeat the same code. Output code only.")


def _new_example(task: dict, target: str, kind: str, source: str, **meta) -> dict:
    prompt = build_initial_messages(task)
    completion = [{"role": "assistant", "content": target}]
    return {
        "task_id": int(task["task_id"]),
        "example_type": kind,
        "source": source,
        "messages": prompt + completion,
        "prompt": prompt,
        "completion": completion,
        "target_code": target,
        **meta,
    }


def _recovery_example(task: dict, failed: dict, correction: str, source: str, sample_index: int) -> dict:
    messages = build_initial_messages(task) + [
        {"role": "assistant", "content": failed["code"]},
        {"role": "user", "content": _retry_text(failed["feedback"])},
        {"role": "assistant", "content": correction},
    ]
    # Conversational prompt-completion format allows Day 3 to train on the
    # CORRECTED assistant reply only, not the failed earlier assistant turn.
    return {
        "task_id": int(task["task_id"]),
        "sample_index": sample_index,
        "example_type": "recovery",
        "source": source,
        "failure_type": failed["status"],
        "failed_code": failed["code"],
        "feedback": failed["feedback"],
        "corrected_code": correction,
        "messages": messages,
        "prompt": messages[:-1],
        "completion": [messages[-1]],
        "target_code": correction,
        "target_turn_only": True,
        "verified_on": "original MBPP visible tests only",
    }


def build_datasets(
    eligible_train: list[dict],
    trajectories: list[dict],
    out_dir: str | Path,
    evalplus_ids: set[int],
    max_success_per_task: int = 2,
    max_recovery_per_task: int = 5,
) -> dict:
    """Verify all selected solutions and export model-vs-reference recovery separately.

    IMPORTANT: Visible-test pass is the only acceptance criterion in Day 2.
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    tasks = {int(t["task_id"]): t for t in eligible_train}
    if set(tasks) & evalplus_ids:
        raise AssertionError("Training examples overlap MBPP+ evaluation IDs")
    invalid_ids = {int(r["task_id"]) for r in trajectories} - set(tasks)
    if invalid_ids:
        raise AssertionError(f"Unexpected/non-training trajectory IDs: {sorted(invalid_ids)}")

    by_task: dict[int, list[dict]] = {task_id: [] for task_id in tasks}
    for tr in trajectories:
        by_task[int(tr["task_id"])].append(tr)

    c1: list[dict] = []
    self_repairs: list[dict] = []
    ref_repairs: list[dict] = []
    reference_failures = []
    unverified_count = 0

    for task_id, task in sorted(tasks.items()):
        histories = sorted(by_task[task_id], key=lambda tr: int(tr["sample_index"]))
        if not histories:
            continue  # only include tasks actually sampled/generated
        seen_success = set()
        seen_repairs = set()
        first_failure: tuple[dict, int] | None = None

        def check(code: str) -> bool:
            return run_visible_tests(code, task["test_list"], task.get("test_setup_code") or "")["status"] == "passed"

        for trajectory in histories:
            sample_idx = int(trajectory["sample_index"])
            hist = trajectory.get("history", [])
            for step in hist:
                if step.get("status") == "passed":
                    code = step.get("code", "").strip()
                    if code and code not in seen_success and len(seen_success) < max_success_per_task:
                        if check(code):
                            seen_success.add(code)
                            c1.append(_new_example(task, code, "success", "model_success",
                                                   sample_index=sample_idx, attempts_used=step["attempt"],
                                                   verified_on="original MBPP visible tests only"))
                        else:
                            unverified_count += 1
                elif step.get("status") in {"syntax_error", "runtime_error", "wrong_output", "timeout"}:
                    if first_failure is None and step.get("code", "").strip():
                        first_failure = (step, sample_idx)

            for previous, nxt in zip(hist, hist[1:]):
                if previous.get("status") in {"passed", "runner_error"} or nxt.get("status") != "passed":
                    continue
                code = nxt.get("code", "").strip()
                prev_code = previous.get("code", "").strip()
                if not code or not prev_code or code == prev_code:
                    continue
                signature = (prev_code, previous.get("feedback", ""), code)
                if signature in seen_repairs or len(seen_repairs) >= max_recovery_per_task:
                    continue
                if check(code):
                    seen_repairs.add(signature)
                    self_repairs.append(_recovery_example(task, previous, code, "self_recovery", sample_idx))
                else:
                    unverified_count += 1

        reference = (task.get("code") or "").strip()
        reference_ok = bool(reference) and check(reference)
        if not seen_success and reference_ok:
            c1.append(_new_example(task, reference, "success", "verified_reference",
                                   sample_index=None, attempts_used=None,
                                   verified_on="original MBPP visible tests only"))
        if not reference_ok:
            reference_failures.append(task_id)
        # Reference correction is a separate, optional experimental data source.
        if first_failure and reference_ok and reference != first_failure[0]["code"].strip():
            ref_repairs.append(_recovery_example(task, first_failure[0], reference,
                                                 "verified_reference_correction", first_failure[1]))

    recovery = self_repairs + ref_repairs
    # Full C2 mixture is provided as a *candidate*. Match actual assistant-target
    # tokens and effective training steps against C1 in the Day 3 training notebook.
    c2_mixed = c1 + recovery
    for rows in [c1, self_repairs, ref_repairs, recovery, c2_mixed]:
        assert not ({int(r["task_id"]) for r in rows} & evalplus_ids)
        assert {int(r["task_id"]) for r in rows} <= set(tasks)

    write_jsonl(out_dir / "success_only.jsonl", c1)
    write_jsonl(out_dir / "recovery_self.jsonl", self_repairs)
    write_jsonl(out_dir / "recovery_reference.jsonl", ref_repairs)
    write_jsonl(out_dir / "recovery.jsonl", recovery)
    write_jsonl(out_dir / "c2_mixed.jsonl", c2_mixed)

    first_failure_counts = Counter()
    first_success = 0
    solved_at_three = 0
    for tr in trajectories:
        if tr.get("history"):
            first_failure_counts[tr["history"][0]["status"]] += 1
            first_success += tr["history"][0]["status"] == "passed"
            solved_at_three += bool(tr.get("solved_visible"))
    summary = {
        "model": trajectories[0]["model"] if trajectories else None,
        "num_eligible_train_tasks": len(tasks),
        "num_trajectories": len(trajectories),
        "unique_sampled_tasks": len({int(tr["task_id"]) for tr in trajectories}),
        "first_attempt_status_counts": dict(first_failure_counts),
        "first_attempt_visible_success_rate": round(first_success / len(trajectories), 4) if trajectories else None,
        "three_attempt_visible_success_rate": round(solved_at_three / len(trajectories), 4) if trajectories else None,
        "c1_success_examples": len(c1),
        "c1_source_counts": dict(Counter(e["source"] for e in c1)),
        "c2_self_recovery_examples": len(self_repairs),
        "c2_reference_recovery_examples_optional": len(ref_repairs),
        "c2_recovery_combined_examples": len(recovery),
        "c2_mixed_candidate_examples": len(c2_mixed),
        "self_recovery_failure_types": dict(Counter(e["failure_type"] for e in self_repairs)),
        "recovery_failure_types_all": dict(Counter(e["failure_type"] for e in recovery)),
        "reference_not_verified_task_ids": reference_failures,
        "rejected_model_candidates_after_recheck": unverified_count,
        "evaluation_overlap_ids": [],
        "notes": [
            "All acceptance checks use ONLY original MBPP visible tests; not hidden MBPP+ correctness.",
            "Correction provenance is recorded; model-generated and reference-based recovery are separated.",
            "Day 3 must use prompt-completion training and match assistant-target token budgets.",
            "Do not use C2 mixed rows naively without token matching/label masking.",
        ],
    }
    (out_dir / "dataset_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    return summary

## D — Re-create the eligible training split (no MBPP+ overlap)

In [ ]:
from prepare_datasets import load_mbpp
from test_runner import run_visible_tests
from day2_dataset import (
    select_training_tasks, collect_trajectories,
    build_datasets, read_jsonl,
)

splits = load_mbpp()  # Google original MBPP full dataset
train_ids = set(manifest['mbppplus_decontaminated']['train']['task_ids'])
evalplus_ids = set(manifest['evalplus_mbppplus_ids'])
validation_ids = set(manifest['validation']['task_ids'])
test_ids = set(manifest['test']['task_ids'])

eligible_train = [r for r in splits['train'] if int(r['task_id']) in train_ids]
assert len(eligible_train) == len(train_ids) == 266
assert train_ids.isdisjoint(evalplus_ids | validation_ids | test_ids)
assert {int(r['task_id']) for r in eligible_train} == train_ids

print('Eligible training tasks:', len(eligible_train))
print('MBPP+ task overlap:', len(train_ids & evalplus_ids))
print('Example task:', eligible_train[0]['task_id'], eligible_train[0]['text'])
print('Visible tests:', eligible_train[0]['test_list'][:2])

# Quick sanity check using a training-only reference solution.
example = eligible_train[0]
ref_result = run_visible_tests(example['code'], example['test_list'], example.get('test_setup_code') or '')
print('Reference visible-test check:', ref_result['status'])

## E — Load Gemma 4 E4B (same model setup as Day 1)

In [ ]:
import torch
import transformers
from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig

MODEL_ID = 'google/gemma-4-E4B-it'
SEED = 42
MAX_NEW_TOKENS = 384
MAX_ATTEMPTS = 3
SAMPLES_PER_TASK = 2
INITIAL_TEMPERATURE = 0.7
RETRY_TEMPERATURE = 0.4
TOP_P = 0.95

torch.manual_seed(SEED)
assert torch.cuda.is_available(), 'Please enable Kaggle GPU accelerator first'
print('GPU:', torch.cuda.get_device_name(0))
print('Transformers:', transformers.__version__)

quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization,
    device_map={'': 0},
    dtype=torch.float16,
)
model.eval()
print('Loaded:', MODEL_ID)
print('GPU 0 memory allocated: %.2f GiB' % (torch.cuda.memory_allocated(0)/1024**3))

## F — Sampled code generation (diverse first attempts, more focused corrections)

In [ ]:
def gemma_generate(messages, max_new_tokens=MAX_NEW_TOKENS):
    inputs = processor.apply_chat_template(
        messages, tokenize=True, return_dict=True, return_tensors='pt',
        add_generation_prompt=True, enable_thinking=False,
    ).to(model.device)
    input_length = int(inputs['input_ids'].shape[1])
    is_retry = len(messages) > 2
    temperature = RETRY_TEMPERATURE if is_retry else INITIAL_TEMPERATURE

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=TOP_P,
            pad_token_id=processor.tokenizer.pad_token_id,
        )
    generated_ids = output[0][input_length:]
    response = processor.decode(generated_ids, skip_special_tokens=True)
    return response, input_length, int(generated_ids.shape[0])

print('Sampled generation ready. Initial temp:', INITIAL_TEMPERATURE,
      '| Retry temp:', RETRY_TEMPERATURE)

## G — Small smoke test: collect 3 training tasks × 2 candidate trajectories

In [ ]:
OUT = Path('/kaggle/working/day2_data')
OUT.mkdir(parents=True, exist_ok=True)
TRAJECTORY_LOG = OUT / 'trajectory_log.jsonl'

SMOKE_N_TASKS = 3
smoke = select_training_tasks(eligible_train, seed=SEED, limit=SMOKE_N_TASKS)

smoke_trajectories = collect_trajectories(
    problems=smoke,
    generate_fn=gemma_generate,
    checkpoint_path=TRAJECTORY_LOG,
    model_id=MODEL_ID,
    seed_fn=torch.manual_seed,
    seed=SEED,
    samples_per_task=SAMPLES_PER_TASK,
    max_attempts=MAX_ATTEMPTS,
    max_new_tokens=MAX_NEW_TOKENS,
)

smoke_summary = build_datasets(
    eligible_train=eligible_train,
    trajectories=smoke_trajectories,
    out_dir=OUT,
    evalplus_ids=evalplus_ids,
)
print(json.dumps({k: v for k, v in smoke_summary.items()
                  if k in ['num_trajectories', 'c1_success_examples',
                           'c2_self_recovery_examples',
                           'c2_reference_recovery_examples_optional']}, indent=2))

**Check the smoke output before continuing.** You should see attempts ending in `passed` or a failure type, plus the number of training examples found. It is completely possible to see **zero** self-recovery examples after only six trajectories. A successful model-generated repair requires a failed attempt followed by an independently passing correction.

## H — Main Day 2 collection (40 training problems × 2 candidate trajectories)

In [ ]:
# Initial manageable batch. If resources remain, increase N_TASKS to 80, 120, or 266
# and rerun this cell. Existing (task_id, sample_index) checkpoints are skipped.
N_TASKS = 40
selected = select_training_tasks(eligible_train, seed=SEED, limit=N_TASKS)

configuration = {
    'model': MODEL_ID, 'seed': SEED, 'num_selected_tasks': N_TASKS,
    'samples_per_task': SAMPLES_PER_TASK,
    'max_attempts': MAX_ATTEMPTS, 'max_new_tokens': MAX_NEW_TOKENS,
    'generation': {'do_sample': True, 'initial_temperature': INITIAL_TEMPERATURE,
                   'retry_temperature': RETRY_TEMPERATURE, 'top_p': TOP_P},
    'evaluation': 'original MBPP visible tests only',
    'train_id_source': 'Day 1 mbpp_split_manifest.json',
}
(OUT / 'data_collection_config.json').write_text(json.dumps(configuration, indent=2))

trajectories = collect_trajectories(
    problems=selected,
    generate_fn=gemma_generate,
    checkpoint_path=TRAJECTORY_LOG,
    model_id=MODEL_ID,
    seed_fn=torch.manual_seed,
    seed=SEED,
    samples_per_task=SAMPLES_PER_TASK,
    max_attempts=MAX_ATTEMPTS,
    max_new_tokens=MAX_NEW_TOKENS,
)
print('Total checkpointed trajectories:', len(trajectories))

## I — Verify corrections and write C1 / C2 training datasets

In [ ]:
# Always rebuild datasets from the persistent trajectory log.
trajectories = read_jsonl(TRAJECTORY_LOG)
summary = build_datasets(
    eligible_train=eligible_train,
    trajectories=trajectories,
    out_dir=OUT,
    evalplus_ids=evalplus_ids,
)
print(json.dumps(summary, indent=2))

print('\nDataset files:')
for name in [
    'success_only.jsonl', 'recovery_self.jsonl',
    'recovery_reference.jsonl', 'recovery.jsonl',
    'c2_mixed.jsonl', 'trajectory_log.jsonl', 'dataset_summary.json'
]:
    path = OUT / name
    print(f'{name}: {path.stat().st_size:,} bytes')

### Dataset interpretation (important for Day 3)

- **`success_only.jsonl` (C1):** Successful solutions verified against visible tests. When the model provides none for a sampled problem, a verified reference solution can fill the gap (source labeled).
- **`recovery_self.jsonl`:** *Genuine Gemma recovery* — a failed attempt immediately followed by a Gemma correction passing visible tests.
- **`recovery_reference.jsonl`:** A model failure paired with the original MBPP reference solution. This is a labeled fallback, **not evidence of model recovery**.
- **`recovery.jsonl`:** Both correction types combined with provenance preserved.
- **`c2_mixed.jsonl` (C2 candidate):** Success examples + recovery examples. **Day 3 must match the C1/C2 assistant-target token budgets and train only on the final assistant completion**, not the failed assistant turn earlier in the prompt. All records contain `prompt` and `completion` specifically for this.

All validation here is on **visible MBPP tests**, not hidden MBPP+ tests. Reference corrections remain available for the optional self-recovery versus reference-correction ablation if time permits.

## J — Inspect a real recovery example and verify dataset boundaries

In [ ]:
self_rows = read_jsonl(OUT / 'recovery_self.jsonl')
reference_rows = read_jsonl(OUT / 'recovery_reference.jsonl')
show = (self_rows or reference_rows)
if show:
    item = show[0]
    print('Task:', item['task_id'], '| Source:', item['source'],
          '| Failure type:', item['failure_type'])
    print('\nFAILED CODE:\n', item['failed_code'][:1500])
    print('\nEXECUTION FEEDBACK:\n', item['feedback'][:900])
    print('\nCORRECTED CODE:\n', item['corrected_code'][:1500])
else:
    print('No correction examples collected yet. Increase N_TASKS / candidate samples.')

for name in ['success_only.jsonl', 'recovery.jsonl', 'c2_mixed.jsonl']:
    rows = read_jsonl(OUT / name)
    assert all(int(r['task_id']) in train_ids for r in rows)
    assert all(int(r['task_id']) not in evalplus_ids for r in rows)
    assert all('prompt' in r and 'completion' in r for r in rows)
    assert all(r['completion'][-1]['role'] == 'assistant' for r in rows)
print('\nData leakage and training-format checks passed.')

## K — Create and download a verified Day 2 backup ZIP

In [ ]:
from zipfile import ZIP_DEFLATED
from IPython.display import FileLink, display

BACKUP = Path('/kaggle/working/gemma_day2_backup.zip')

# Include complete raw trajectories, training datasets, manifest, and source code.
backup_files = [p for p in sorted(OUT.iterdir()) if p.is_file()]
backup_files += [SRC / x for x in [
    'prepare_datasets.py', 'test_runner.py', 'agent_loop.py',
    'day2_dataset.py', 'mbpp_split_manifest.json'
]]
missing = [str(p) for p in backup_files if not p.is_file()]
if missing:
    raise FileNotFoundError('Missing backup files: ' + str(missing))

with ZipFile(BACKUP, 'w', ZIP_DEFLATED, compresslevel=3) as archive:
    for path in backup_files:
        archive.write(path, arcname=('day2_data' if path.parent == OUT else 'day2_src') + '/' + path.name)
with ZipFile(BACKUP) as archive:
    assert archive.testzip() is None, 'Backup integrity check failed'
    assert len(archive.namelist()) == len(backup_files)
    print('Verified backup containing', len(archive.namelist()), 'files')
    print('ZIP:', BACKUP, '| Size:', round(BACKUP.stat().st_size/1024, 1), 'KB')
display(FileLink(str(BACKUP)))

## After Day 2

1. Download `gemma_day2_backup.zip` from the last cell while the session is still running.
2. **Save Version → Quick Save**, including outputs where available; check the version's **Output** tab.
3. Keep Day 1 and Day 2 versions separate. Do **not** generate or use validation/test examples for training.
4. For Day 3, attach this notebook's saved outputs to a new Kaggle fine-tuning notebook, then train C1 and C2 with comparable **assistant-target token** budgets.

**Security note:** Executing model-generated Python in a subprocess with timeouts is not equivalent to true isolation. Do not use this runner for untrusted external submissions or in an environment containing secrets.